# Driver Drowsiness Detection — Honest Subject-Level Evaluation

MobileNetV2 transfer learning on the Driver Drowsiness Dataset (DDD), evaluated
under a strict **subject-disjoint** split to eliminate identity leakage.

**Pipeline order:** run cells top to bottom. Training weights are checkpointed to
Google Drive, so after a runtime reset you can reload instead of retraining
(see the *Reload after reset* cell).

**Result summary:** frozen-base validation AUC 0.731; fine-tuning degrades
generalization; held-out test AUC 0.412 (below chance) with 38% of drowsy frames
missed. Per-subject test AUC ranges 0.003–0.721, indicating no transferable
drowsiness cue was learned.

## 1. Setup and data

In [ ]:
# 1.1 — Environment check (confirm T4 GPU is active)
import tensorflow as tf
print("TF version:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices('GPU'))

In [ ]:
# 1.2 — Download DDD from Kaggle
import kagglehub
DATA_ROOT = kagglehub.dataset_download("ismailnasri20/driver-drowsiness-dataset-ddd")
print("Downloaded to:", DATA_ROOT)

In [ ]:
# 1.3 — Verify folder structure and image counts
import os

BASE_DIR = os.path.join(DATA_ROOT, "Driver Drowsiness Dataset (DDD)")
if not os.path.isdir(BASE_DIR):
    for root, dirs, _ in os.walk(DATA_ROOT):
        if "Drowsy" in dirs and "Non Drowsy" in dirs:
            BASE_DIR = root
            break

CLASSES = ["Drowsy", "Non Drowsy"]
print("BASE_DIR:", BASE_DIR)
for c in CLASSES:
    p = os.path.join(BASE_DIR, c)
    n = len([f for f in os.listdir(p) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
    print(f"{c}: {n} images")

## 2. Leakage diagnosis and subject-disjoint split

Filenames carry a leading letter prefix (e.g. `A0001.png`) that identifies the
subject/recording. Treated case-insensitively there are 28 subjects, 26 of which
appear in **both** classes. A naive split therefore leaks identity across
train/val/test. We split by whole subject so no person crosses a boundary.

In [ ]:
# 2.1 — Master index: filepath, subject, label (Drowsy=1, Non Drowsy=0)
import pandas as pd, re, os

def subject_of(fname):
    m = re.match(r"^([A-Za-z]+)", fname)
    return m.group(1).upper() if m else "<none>"   # uppercase = case-insensitive subject

LABEL_MAP = {"Drowsy": 1, "Non Drowsy": 0}          # positive class = Drowsy (safety-critical)

rows = []
for c in CLASSES:
    cdir = os.path.join(BASE_DIR, c)
    for f in os.listdir(cdir):
        if f.lower().endswith(('.png', '.jpg', '.jpeg')):
            rows.append({"filepath": os.path.join(cdir, f),
                         "subject": subject_of(f),
                         "label": LABEL_MAP[c],
                         "class_name": c})

df = pd.DataFrame(rows)
print("Total images:", len(df))
print(df["class_name"].value_counts())
print("Distinct subjects:", df["subject"].nunique())

In [ ]:
# 2.2 — Curated subject-level split (leak-free) + verification
TEST_SUBJECTS = {"E", "H", "Q", "S", "U"}
VAL_SUBJECTS  = {"C", "J", "K", "N", "R"}
# everything else -> train (includes single-class subjects F, T)

def assign_split(s):
    if s in TEST_SUBJECTS: return "test"
    if s in VAL_SUBJECTS:  return "val"
    return "train"

df["split"] = df["subject"].apply(assign_split)

# Verification 1 — no subject spans more than one split
overlap = df.groupby("subject")["split"].nunique()
assert overlap.max() == 1, "LEAK: a subject spans multiple splits!"
print("PASS: no subject spans multiple splits.\n")

# Verification 2 — per-split sizes and class balance
summary = df.groupby(["split", "class_name"]).size().unstack(fill_value=0)
summary["total"] = summary.sum(axis=1)
summary["%"] = (100 * summary["total"] / len(df)).round(1)
print(summary, "\n")

for sp in ["train", "val", "test"]:
    subs = sorted(df.loc[df.split == sp, "subject"].unique())
    print(f"{sp:5s} ({len(subs)} subjects): {subs}")

In [ ]:
# 2.3 — tf.data pipelines with MobileNetV2 preprocessing
import tensorflow as tf
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

IMG_SIZE, BATCH, AUTOTUNE = 224, 32, tf.data.AUTOTUNE

def make_ds(split, training):
    sub = df[df.split == split]
    paths, labels = sub["filepath"].values, sub["label"].values.astype("float32")
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(len(paths), seed=42, reshuffle_each_iteration=True)

    def load(path, label):
        img = tf.io.decode_png(tf.io.read_file(path), channels=3)
        img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
        return img, label                                  # float32 in [0,255]
    ds = ds.map(load, num_parallel_calls=AUTOTUNE)

    if training:
        def augment(img, label):
            img = tf.image.random_flip_left_right(img)
            img = tf.image.random_brightness(img, 0.15 * 255)
            img = tf.image.random_contrast(img, 0.85, 1.15)
            return tf.clip_by_value(img, 0.0, 255.0), label
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)

    ds = ds.map(lambda x, y: (preprocess_input(x), y), num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH).prefetch(AUTOTUNE)

train_ds = make_ds("train", True)
val_ds   = make_ds("val",   False)
test_ds  = make_ds("test",  False)

xb, yb = next(iter(train_ds))
print("Batch:", xb.shape, "| pixel range:",
      round(float(xb.numpy().min()), 2), "to", round(float(xb.numpy().max()), 2))

In [ ]:
# 2.4 — Class weights from the TRAIN split only
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

train_labels = df.loc[df.split == "train", "label"].values
weights = compute_class_weight("balanced", classes=np.array([0, 1]), y=train_labels)
class_weight = {0: float(weights[0]), 1: float(weights[1])}
print("Non Drowsy:", int((train_labels == 0).sum()),
      "| Drowsy:", int((train_labels == 1).sum()))
print("class_weight:", class_weight)

## 3. Model and Stage 1 training (frozen base)

MobileNetV2 (ImageNet weights, classifier removed) with a small head. Stage 1
freezes the backbone and trains only the head. Checkpoints save to Drive so a
runtime reset does not force retraining.

In [ ]:
# 3.0 — Mount Drive so checkpoints survive resets
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/drowsiness', exist_ok=True)
print("Drive mounted.")

In [ ]:
# 3.1 — Build model: frozen MobileNetV2 base + fresh head
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV2

tf.random.set_seed(42)

base = MobileNetV2(include_top=False, weights="imagenet", input_shape=(224, 224, 3))
base.trainable = False                       # Stage 1: freeze everything

inputs = tf.keras.Input(shape=(224, 224, 3))
x = base(inputs, training=False)             # keep BatchNorm in inference mode
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)   # P(Drowsy)
model = Model(inputs, outputs)

print("Total params:    ", f"{model.count_params():,}")
print("Trainable params:", f"{sum(tf.size(w).numpy() for w in model.trainable_weights):,}")

In [ ]:
# 3.2 — Compile + callbacks
from tensorflow.keras.metrics import Precision, Recall, AUC
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy", Precision(name="precision"),
             Recall(name="recall"), AUC(name="auc")],
)

CKPT = "/content/drive/MyDrive/drowsiness/stage1_best.weights.h5"
callbacks = [
    EarlyStopping(monitor="val_auc", mode="max", patience=4, restore_best_weights=True),
    ModelCheckpoint(CKPT, monitor="val_auc", mode="max",
                    save_best_only=True, save_weights_only=True),
]
print("Compiled. Best weights ->", CKPT)

In [ ]:
# 3.3 — Train Stage 1 (frozen base)
# Skip this cell after a reset if weights already exist on Drive; use 3.4 instead.
EPOCHS_STAGE1 = 12
history1 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=EPOCHS_STAGE1,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1,
)
print("\nStage 1 done. Best val_auc: %.4f" % max(history1.history["val_auc"]))

### Reload after reset

If the runtime disconnects, run cells **1.2, 1.3, 2.1–2.4, 3.0, 3.1, 3.2**, then
this cell to restore the trained model from Drive — no retraining needed.

In [ ]:
# 3.4 — Reload the final (Stage-1) model from Drive
model.load_weights("/content/drive/MyDrive/drowsiness/stage1_best.weights.h5")
loss, acc, prec, rec, auc = model.evaluate(val_ds, verbose=0)
print(f"Final model loaded. val_auc={auc:.4f} (expect ~0.731)")

## 4. Fine-tuning experiments (negative result — retained for reproducibility)

Unfreezing the top layers and fine-tuning **degraded** validation AUC
(0.731 → 0.599 with 30 layers; → 0.697 with a gentler 4-layer retry). This
confirms the added capacity is spent memorizing subject identity rather than
learning a transferable cue. **Stage 1 remains the final model.** These cells are
kept only to document the experiment; skip them for the final pipeline.

In [ ]:
# 4.1 — [Fine-tuning experiment] Unfreeze top 30 layers (BatchNorm frozen)
# Optional. Degrades generalization; retained for documentation only.
base.trainable = True
N_UNFREEZE = 30

n = len(base.layers)
for i, layer in enumerate(base.layers):
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False
    else:
        layer.trainable = (i >= n - N_UNFREEZE)

print(f"Base has {n} layers | unfreezing top {N_UNFREEZE} (BatchNorm frozen)")
print("Trainable params:",
      f"{sum(tf.size(w).numpy() for w in model.trainable_weights):,}")

In [ ]:
# 4.2 — [Fine-tuning experiment] Recompile at low LR and fine-tune
# Optional. Result: val_auc ~0.599 (worse than Stage 1). Do NOT overwrite the
# Stage-1 checkpoint; a separate path is used.
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy", Precision(name="precision"),
             Recall(name="recall"), AUC(name="auc")],
)
CKPT2 = "/content/drive/MyDrive/drowsiness/stage2_best.weights.h5"
callbacks2 = [
    EarlyStopping(monitor="val_auc", mode="max", patience=4, restore_best_weights=True),
    ModelCheckpoint(CKPT2, monitor="val_auc", mode="max",
                    save_best_only=True, save_weights_only=True),
]
history2 = model.fit(
    train_ds, validation_data=val_ds, epochs=10,
    class_weight=class_weight, callbacks=callbacks2, verbose=1,
)
print("\nStage 2 best val_auc: %.4f (Stage 1 was 0.7310)"
      % max(history2.history["val_auc"]))

In [ ]:
# 4.3 — [Fine-tuning experiment] Restore Stage-1 as final model
# Run this after the experiment to discard the degraded Stage-2 weights.
model.load_weights("/content/drive/MyDrive/drowsiness/stage1_best.weights.h5")
print("Restored Stage-1 weights as the final model.")

## 5. Held-out test evaluation

The test split (subjects E, H, Q, S, U) is evaluated once. Predictions are cached
here and reused by every downstream analysis.

In [ ]:
# 5.1 — Held-out TEST predictions (compute once, reuse everywhere)
import numpy as np

test_sub = df[df.split == "test"].reset_index(drop=True)
paths_arr  = test_sub["filepath"].values
labels_arr = test_sub["label"].values.astype("float32")

eval_ds = tf.data.Dataset.from_tensor_slices((paths_arr, labels_arr))
def load_eval(path, label):
    img = tf.io.decode_png(tf.io.read_file(path), channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    return preprocess_input(img), label
eval_ds = eval_ds.map(load_eval, num_parallel_calls=AUTOTUNE).batch(BATCH).prefetch(AUTOTUNE)

y_prob = model.predict(eval_ds, verbose=1).ravel()   # P(Drowsy) per image
y_true = labels_arr.astype(int)
paths_test = paths_arr

print("Test images:", len(y_true))
print("Prob range:", round(float(y_prob.min()), 3), "to", round(float(y_prob.max()), 3))
print("Mean P(Drowsy) — true Drowsy:", round(float(y_prob[y_true==1].mean()), 3),
      "| true Non-Drowsy:", round(float(y_prob[y_true==0].mean()), 3))

In [ ]:
# 5.2 — Test AUC
from sklearn.metrics import roc_auc_score
test_auc = roc_auc_score(y_true, y_prob)
print(f"Test AUC (Drowsy as positive): {test_auc:.4f}")

In [ ]:
# 5.3 — Confusion matrix + per-class metrics @ threshold 0.5
from sklearn.metrics import confusion_matrix, classification_report

THRESH = 0.5
y_pred = (y_prob >= THRESH).astype(int)
cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()

print("Confusion matrix @0.5  (rows=true, cols=pred)")
print(f"                 pred_NonDrowsy   pred_Drowsy")
print(f"true_NonDrowsy   {tn:>12}   {fp:>11}")
print(f"true_Drowsy      {fn:>12}   {tp:>11}")
print(f"\nMissed drowsy (FN): {fn}  ({100*fn/(fn+tp):.1f}% of all drowsy)")
print(f"False alarms  (FP): {fp}  ({100*fp/(fp+tn):.1f}% of all non-drowsy)")
print("\nPer-class metrics @0.5:")
print(classification_report(y_true, y_pred, target_names=["Non Drowsy", "Drowsy"], digits=3))
print(f"Overall accuracy @0.5: {(tp+tn)/len(y_true):.3f}")

## 6. Error analysis and interpretability

Threshold analysis, per-subject breakdown, confidence analysis, and Grad-CAM.
Answers: *where, why, and how confidently does the model fail?*

In [ ]:
# 6.1 — ROC + Precision-Recall curves  ->  saves roc_pr.png
import numpy as np, matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, precision_recall_curve, roc_auc_score, average_precision_score

roc_auc   = roc_auc_score(y_true, y_prob)
ap        = average_precision_score(y_true, y_prob)
base_rate = y_true.mean()
fpr, tpr, _ = roc_curve(y_true, y_prob)
prec, rec, _ = precision_recall_curve(y_true, y_prob)

fig, ax = plt.subplots(1, 2, figsize=(8, 3.4))
ax[0].plot(fpr, tpr, color="C0", lw=2, label=f"Model (AUC={roc_auc:.3f})")
ax[0].plot([0,1],[0,1], "--", color="gray", label="Chance (0.5)")
ax[0].set_xlabel("False Positive Rate"); ax[0].set_ylabel("True Positive Rate")
ax[0].set_title("ROC — Drowsy"); ax[0].legend(fontsize=8); ax[0].set_aspect("equal")

ax[1].plot(rec, prec, color="C1", lw=2, label=f"Model (AP={ap:.3f})")
ax[1].axhline(base_rate, ls="--", color="gray", label=f"No-skill ({base_rate:.3f})")
ax[1].set_xlabel("Recall (Drowsy)"); ax[1].set_ylabel("Precision (Drowsy)")
ax[1].set_title("Precision-Recall — Drowsy"); ax[1].legend(fontsize=8); ax[1].set_ylim(0, 1.02)

plt.tight_layout()
plt.savefig("/content/drive/MyDrive/drowsiness/roc_pr.png", dpi=200, bbox_inches="tight")
plt.show()
print(f"ROC-AUC: {roc_auc:.4f} | AP: {ap:.4f} | base rate: {base_rate:.3f}")

In [ ]:
# 6.2 — Best achievable precision at each target recall (threshold analysis)
import numpy as np
print(f"{'target recall':>14} | {'best precision':>14} | {'threshold':>10}")
print("-" * 44)
prec, rec, thr = precision_recall_curve(y_true, y_prob)
for R in [0.60, 0.70, 0.80, 0.90, 0.95, 0.99]:
    mask = rec[:-1] >= R
    if mask.sum() > 0:
        idxs = np.where(mask)[0]
        bi = idxs[np.argmax(prec[:-1][idxs])]
        print(f"{R:>14.2f} | {prec[bi]:>14.3f} | {thr[bi]:>10.3f}")
print(f"\nBase-rate (random) precision = {y_true.mean():.3f}")

In [ ]:
# 6.3 — Per-subject test breakdown  ->  saves persubject_auc.png
import numpy as np, matplotlib.pyplot as plt, re, os
from sklearn.metrics import roc_auc_score

def subj_of_path(p):
    m = re.match(r"^([A-Za-z]+)", os.path.basename(p)); return m.group(1).upper() if m else "?"
subj_arr = np.array([subj_of_path(p) for p in paths_test])
y_pred = (y_prob >= 0.5).astype(int)

print(f"{'subj':>4} | {'n':>5} | {'AUC':>6} | {'acc':>6} | {'drowsy_recall':>13} | {'mean_P':>7}")
print("-" * 60)
rows = []
for s in sorted(set(subj_arr)):
    m = subj_arr == s
    yt, yp, prob = y_true[m], y_pred[m], y_prob[m]
    auc = roc_auc_score(yt, prob) if len(set(yt)) == 2 else float("nan")
    dr  = yp[yt == 1].mean() if (yt == 1).any() else float("nan")
    print(f"{s:>4} | {m.sum():>5} | {auc:>6.3f} | {(yt==yp).mean():>6.3f} | "
          f"{dr:>13.3f} | {prob.mean():>7.3f}")
    if not np.isnan(auc): rows.append((s, auc))

rows.sort(key=lambda r: r[1])
subs = [r[0] for r in rows]; aucs = [r[1] for r in rows]
colors = ["#c0392b" if a < 0.5 else "#2980b9" for a in aucs]

fig, ax = plt.subplots(figsize=(6, 3.4))
bars = ax.bar(subs, aucs, color=colors)
ax.axhline(0.5, ls="--", color="gray", lw=1.2, label="Chance (0.5)")
ax.axhline(roc_auc, ls=":", color="black", lw=1.2, label=f"Overall ({roc_auc:.3f})")
ax.set_xlabel("Test Subject"); ax.set_ylabel("ROC-AUC (Drowsy)")
ax.set_title("Per-Subject Test AUC"); ax.set_ylim(0, 0.8); ax.legend(fontsize=8)
for b, a in zip(bars, aucs):
    ax.text(b.get_x()+b.get_width()/2, a+0.015, f"{a:.3f}", ha="center", fontsize=8)
plt.tight_layout()
plt.savefig("/content/drive/MyDrive/drowsiness/persubject_auc.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# 6.4 — Confidence analysis: most confidently wrong cases
import numpy as np
y_pred = (y_prob >= 0.5).astype(int)

fn_idx = np.where((y_true == 1) & (y_pred == 0))[0]
fp_idx = np.where((y_true == 0) & (y_pred == 1))[0]
worst_fn = fn_idx[np.argsort(y_prob[fn_idx])][:5]        # lowest P = most confidently "alert"
worst_fp = fp_idx[np.argsort(-y_prob[fp_idx])][:5]       # highest P = most confidently "drowsy"

print("Mean P(Drowsy) — missed drowsy (FN):", round(float(y_prob[fn_idx].mean()), 3))
print("Mean P(Drowsy) — false alarms  (FP):", round(float(y_prob[fp_idx].mean()), 3))
print("\n5 most confident MISSED drowsy:")
for i in worst_fn:
    print(f"   subj {subj_arr[i]:>3}  P(Drowsy)={y_prob[i]:.3f}")
print("5 most confident FALSE alarms:")
for i in worst_fp:
    print(f"   subj {subj_arr[i]:>3}  P(Drowsy)={y_prob[i]:.3f}")

### Grad-CAM interpretability

In [ ]:
# 6.5 — Grad-CAM setup
import numpy as np, tensorflow as tf
LAST_CONV = "Conv_1"   # MobileNetV2 final conv layer

def make_gradcam_heatmap(img_array, model, last_conv_name=LAST_CONV):
    base_model = None
    for layer in model.layers:
        if isinstance(layer, tf.keras.Model):
            base_model = layer; break
    conv_layer = base_model.get_layer(last_conv_name)
    grad_model = tf.keras.models.Model([base_model.inputs], [conv_layer.output, base_model.output])
    with tf.GradientTape() as tape:
        conv_out, base_out = grad_model(img_array)
        x = tf.keras.layers.GlobalAveragePooling2D()(base_out)
        score = model.layers[-1](x)
    grads = tape.gradient(score, conv_out)
    pooled = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_out = conv_out[0]
    heatmap = tf.reduce_sum(conv_out * pooled, axis=-1)
    heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy(), float(score.numpy().ravel()[0])

print("Grad-CAM ready.")

In [ ]:
# 6.6 — Region-attention summary across outcome categories
import numpy as np
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

def region_attention(hm):
    total = hm.sum() + 1e-8
    return {
        "top (hair/forehead)":    100*hm[0:2, :].sum()/total,
        "upper-mid (EYES)":       100*hm[2:4, 1:6].sum()/total,
        "lower-mid (nose/mouth)": 100*hm[4:5, 1:6].sum()/total,
        "bottom (chin/neck)":     100*hm[5:7, :].sum()/total,
        "left edge (bg/ear)":     100*hm[2:5, 0:1].sum()/total,
        "right edge (bg/ear)":    100*hm[2:5, 6:7].sum()/total,
    }

def load_one(path):
    img = tf.image.resize(tf.io.decode_png(tf.io.read_file(path), channels=3), [224, 224])
    return preprocess_input(img)[None, ...]

y_pred = (y_prob >= 0.5).astype(int)
categories = {
    "Correct Drowsy (TP)":    np.where((y_true==1)&(y_pred==1))[0],
    "Missed Drowsy (FN)":     np.where((y_true==1)&(y_pred==0))[0],
    "Correct NonDrowsy (TN)": np.where((y_true==0)&(y_pred==0))[0],
    "False alarm (FP)":       np.where((y_true==0)&(y_pred==1))[0],
}
region_keys = ["top (hair/forehead)", "upper-mid (EYES)", "lower-mid (nose/mouth)",
               "bottom (chin/neck)", "left edge (bg/ear)", "right edge (bg/ear)"]
rng = np.random.default_rng(42)
N_PER = 40

for cat, idxs in categories.items():
    if len(idxs) == 0: continue
    pick = rng.choice(idxs, size=min(N_PER, len(idxs)), replace=False)
    acc = np.zeros(len(region_keys)); confs = []
    for i in pick:
        hm, score = make_gradcam_heatmap(load_one(paths_test[i]), model)
        r = region_attention(hm); acc += np.array([r[k] for k in region_keys]); confs.append(score)
    acc /= len(pick)
    print(f"=== {cat}  (n={len(pick)}, mean P(Drowsy)={np.mean(confs):.3f}) ===")
    for k, v in zip(region_keys, acc):
        print(f"   {k:<24}: {v:5.1f}%")
    print()

In [ ]:
# 6.7 — Grad-CAM overlays for representative cases  ->  saves gradcam_compact.png
import numpy as np, matplotlib.pyplot as plt, matplotlib.cm as cm
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

y_pred = (y_prob >= 0.5).astype(int)

def load_raw_pre(path):
    raw = tf.image.resize(tf.io.decode_png(tf.io.read_file(path), channels=3), [224,224])
    return raw.numpy().astype("uint8"), preprocess_input(tf.identity(raw))[None,...]

def overlay(raw, hm, alpha=0.45):
    h = tf.image.resize((hm[...,None]*255).astype("uint8"), [224,224]).numpy().squeeze()
    h = h/(h.max()+1e-8); col = cm.jet(h)[...,:3]
    return np.clip((1-alpha)*(raw/255.0)+alpha*col, 0, 1)

tp   = np.where((y_true==1)&(y_pred==1))[0][0]
fn_H = np.where((y_true==1)&(y_pred==0)&(subj_arr=="H"))[0]
fp_Q = np.where((y_true==0)&(y_pred==1)&(subj_arr=="Q"))[0]
picks = [("Correct Drowsy", tp)]
if len(fn_H): picks.append(("Missed Drowsy (H)", fn_H[0]))
if len(fp_Q): picks.append(("False Alarm (Q)", fp_Q[0]))

fig, axes = plt.subplots(len(picks), 2, figsize=(4.5, 2.2*len(picks)))
if len(picks) == 1: axes = axes[None, :]
for r,(tag,i) in enumerate(picks):
    raw, pre = load_raw_pre(paths_test[i])
    hm, score = make_gradcam_heatmap(pre, model)
    truth = "Drowsy" if y_true[i]==1 else "Alert"
    axes[r,0].imshow(raw); axes[r,0].set_title(f"{tag}\ntrue={truth}", fontsize=9)
    axes[r,1].imshow(overlay(raw,hm)); axes[r,1].set_title(f"Grad-CAM  P(D)={score:.2f}", fontsize=9)
    for c in range(2): axes[r,c].axis("off")
plt.tight_layout()
plt.savefig("/content/drive/MyDrive/drowsiness/gradcam_compact.png", dpi=200, bbox_inches="tight")
plt.show()

## 7. Summary

| Stage | Metric | Value |
|---|---|---|
| Leaky baseline (prior work) | Val accuracy | ~0.76 |
| Stage 1 (frozen base) | Val AUC | 0.731 |
| Stage 2 fine-tune (30 / 4 layers) | Val AUC | 0.599 / 0.697 |
| **Held-out test (final)** | **AUC** | **0.412** |
| Held-out test | Accuracy @0.5 | 0.526 |
| Held-out test | Missed drowsy (FN) | 38.0% |
| Per-subject test AUC | Range | 0.003 – 0.721 |

**Conclusion.** Under leak-free subject-disjoint evaluation, MobileNetV2 transfer
learning on DDD does not learn a generalizable drowsiness cue. The failure is
primarily a dataset limitation (few subjects; single-frame inputs lacking
temporal cues), secondarily an architecture–task mismatch, and not a
methodological one — the rigorous evaluation is what exposes the true limitation.
Recommended directions: eye-region cropping, more subjects, and temporal
modeling.